# 02. Limpieza y organización — Dataset Bancario Empresarial

En este notebook se realizan las transformaciones necesarias para limpiar y estandarizar el dataset identificado durante el EDA. El objetivo es obtener un conjunto de datos consistente.

Este notebook No modifica ni sobreescribe el archivo original dataset_bancario_empresarial.xlsx. El script lee la fuente cruda, procesa en memoria y exporta el resultado en dataset_limpio.xlsx.

**Hallazgos de partida (del EDA):**


*Formato de columnas*

- Formatear las columnas `fecha_operacion`, `fecha_contable` y `fecha_vencimiento` al tipo datetime64 (Formato AAAA-MM-DD)
- Columna `requiere_revision` viene como texto "Si"/"No", convertir a verdadero/falso

*Tratar nulos según significado*

- `motivo_revision` Nulo = estado esperado
- `observaciones` "" Campo libre opcional
- Falta de asignación interna `responsable`, `centro_costo` quedan "Sin asignar"
- Falta de captura en el momento de la transacción: `proveedor_cliente`, `documento_ref` se rellenan como "Sin registro"

*Estandarizar textos*

- Quitar espacios al inicio/final
- Unificar mayúsculas/minúsculas

*Validar reglas bancarias*

- Si `tipo_movimiento` == "Ingreso", el valor debería ser positivo.
- Si `tipo_movimiento` == "Egreso", el valor debería ser negativo.
- `valor_neto` debería coincidir con `valor_bruto` + `impuesto_iva`.
- `fecha_contable` no debería ser anterior a `fecha_operacion`.
- Si `estado_conciliacion` == "Conciliado", normalmente dias_mora debería ser 0.

*Alerta de Negocio: Estados Duplicados*

El EDA mostró **4 registros** marcados como `Duplicado` en la columna `estado_conciliacion`. 

**Cuidado:** Esto es distinto a los duplicados de fila tradicionales. Es un **estado de negocio** que ya fue identificado y debemos separar para revisión.
`estado_conciliacion == 'Duplicado'

In [1]:
# Importar librerías
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [5]:
# Cargar el dataset ORIGINAL (no se sobreescribe)
RUTA_RAW = "dataset_bancario_empresarial.xlsx"
df = pd.read_excel(RUTA_RAW)

In [6]:
# Copia de trabajo: el df original queda intacto para comparar si hace falta
df_limpio = df.copy()

# 1. Confirmar formato de columnas

In [7]:
# Fechas -> datetime64 (AAAA-MM-DD)
columnas_fecha = ["fecha_operacion", "fecha_contable", "fecha_vencimiento"]
for col in columnas_fecha:
    df_limpio[col] = pd.to_datetime(df_limpio[col]).dt.normalize()

df_limpio[columnas_fecha].dtypes

fecha_operacion      datetime64[ns]
fecha_contable       datetime64[ns]
fecha_vencimiento    datetime64[ns]
dtype: object

In [8]:
# requiere_revision: texto "Si"/"No" -> booleano
print("Valores originales:", df_limpio["requiere_revision"].unique())

df_limpio["requiere_revision"] = (
    df_limpio["requiere_revision"].astype(str).str.strip().str.lower().eq("si")
)

df_limpio["requiere_revision"].value_counts()

Valores originales: ['No' 'Si']


requiere_revision
True     84
False    66
Name: count, dtype: int64

# 2. Tratar nulos según significado

In [9]:
# Nulos antes de tratar (referencia)
df_limpio.isnull().sum()[df_limpio.isnull().sum() > 0]

proveedor_cliente     13
documento_ref          6
centro_costo          29
responsable           34
observaciones        136
motivo_revision       66
dtype: int64

In [10]:
# motivo_revision: nulo = estado esperado (no requería revisión)
df_limpio["motivo_revision"] = df_limpio["motivo_revision"].fillna("No aplica")

# observaciones: campo libre opcional
df_limpio["observaciones"] = df_limpio["observaciones"].fillna("")

# responsable, centro_costo: falta de asignación interna
df_limpio["responsable"] = df_limpio["responsable"].fillna("Sin asignar")
df_limpio["centro_costo"] = df_limpio["centro_costo"].fillna("Sin asignar")

# proveedor_cliente, documento_ref: falta de captura al momento de la transacción
df_limpio["proveedor_cliente"] = df_limpio["proveedor_cliente"].fillna("Sin registro")
df_limpio["documento_ref"] = df_limpio["documento_ref"].fillna("Sin registro")

In [11]:
# Verificación: no deberían quedar nulos
nulos_restantes = df_limpio.isnull().sum().sum()
print(f"Nulos restantes en el dataset: {nulos_restantes}")

Nulos restantes en el dataset: 0


# 3. Estandarizar textos

In [12]:
columnas_categoricas = [
    "empresa", "banco", "tipo_movimiento", "categoria", "moneda",
    "metodo_pago", "centro_costo", "ciudad", "pais", "responsable",
    "estado_conciliacion", "nivel_riesgo", "canal",
]

# Quitar espacios al inicio/final
for col in columnas_categoricas:
    df_limpio[col] = df_limpio[col].astype(str).str.strip()

# Revisión de valores únicos por columna (para detectar variantes tipo "Bogota " vs "bogota")
for col in ["ciudad", "banco", "categoria"]:
    print(col, "->", sorted(df_limpio[col].unique()))

ciudad -> ['Barranquilla', 'Bogota', 'Cali', 'Cartagena', 'Medellin']
banco -> ['Banco Andino', 'Banco Capital', 'Banco Nacional', 'Banco Pacifico']
categoria -> ['Arriendo', 'Comision bancaria', 'Credito bancario', 'Impuestos', 'Nomina', 'Pago proveedor', 'Reembolso', 'Servicios publicos', 'Venta internacional', 'Venta nacional']


## 4. Validar reglas bancarias

Estas reglas **no corrigen los datos** — solo identifican qué registros las incumplen, para decidir caso por caso si son errores de captura o excepciones válidas.

In [13]:
# Regla 1 y 2: signo de valor_bruto según tipo_movimiento
esperado_negativo = df_limpio["tipo_movimiento"] == "Egreso"
es_negativo = df_limpio["valor_bruto"] < 0

df_limpio["regla_signo_valida"] = esperado_negativo == es_negativo
anomalias_signo = df_limpio[~df_limpio["regla_signo_valida"]]
print(f"Transacciones con signo inconsistente: {len(anomalias_signo)}")
anomalias_signo[["transaccion_id", "tipo_movimiento", "valor_bruto"]]

Transacciones con signo inconsistente: 0


,transaccion_id,tipo_movimiento,valor_bruto


In [14]:
# Regla 3: valor_neto == valor_bruto + impuesto_iva
diferencia = (df_limpio["valor_neto"] - (df_limpio["valor_bruto"] + df_limpio["impuesto_iva"])).abs()
df_limpio["regla_valor_neto_valida"] = diferencia < 0.01  # tolerancia por redondeo

anomalias_valor_neto = df_limpio[~df_limpio["regla_valor_neto_valida"]]
print(f"Transacciones con valor_neto inconsistente: {len(anomalias_valor_neto)}")
anomalias_valor_neto[["transaccion_id", "valor_bruto", "impuesto_iva", "valor_neto"]]

Transacciones con valor_neto inconsistente: 0


,transaccion_id,valor_bruto,impuesto_iva,valor_neto


In [15]:
# Regla 4: fecha_contable no debe ser anterior a fecha_operacion
df_limpio["regla_fechas_valida"] = df_limpio["fecha_contable"] >= df_limpio["fecha_operacion"]

anomalias_fechas = df_limpio[~df_limpio["regla_fechas_valida"]]
print(f"Transacciones con fecha_contable anterior a fecha_operacion: {len(anomalias_fechas)}")
anomalias_fechas[["transaccion_id", "fecha_operacion", "fecha_contable"]]

Transacciones con fecha_contable anterior a fecha_operacion: 0


,transaccion_id,fecha_operacion,fecha_contable


In [16]:
# Regla 5: si esta Conciliado, dias_mora debería ser 0
conciliado_sin_mora = ~(
    (df_limpio["estado_conciliacion"] == "Conciliado") & (df_limpio["dias_mora"] != 0)
)
df_limpio["regla_mora_valida"] = conciliado_sin_mora

anomalias_mora = df_limpio[~df_limpio["regla_mora_valida"]]
print(f"Transacciones Conciliadas con dias_mora distinto de 0: {len(anomalias_mora)}")
anomalias_mora[["transaccion_id", "estado_conciliacion", "dias_mora"]]

Transacciones Conciliadas con dias_mora distinto de 0: 0


,transaccion_id,estado_conciliacion,dias_mora


## 5. Resumen de anomalías

Consolidado de las 5 reglas del punto anterior. No se corrige nada aquí — es el mapa para decidir qué revisar manualmente.

In [17]:
resumen_anomalias = pd.DataFrame({
    "regla": [
        "Signo valor_bruto vs tipo_movimiento",
        "valor_neto = valor_bruto + impuesto_iva",
        "fecha_contable >= fecha_operacion",
        "Conciliado -> dias_mora = 0",
    ],
    "transacciones_con_anomalia": [
        (~df_limpio["regla_signo_valida"]).sum(),
        (~df_limpio["regla_valor_neto_valida"]).sum(),
        (~df_limpio["regla_fechas_valida"]).sum(),
        (~df_limpio["regla_mora_valida"]).sum(),
    ],
})
resumen_anomalias

,regla,transacciones_con_anomalia
0,Signo valor_bruto vs tipo_movimiento,0
1,valor_neto = valor_bruto + impuesto_iva,0
2,fecha_contable >= fecha_operacion,0
3,Conciliado -> dias_mora = 0,0


**Decisión:** no se encontraron anomalías en ninguna regla de negocio, el dataset queda validado.


# 6 Alerta de Negocio: Estados Duplicados

In [18]:
posibles_duplicados_negocio = df_limpio[df_limpio["estado_conciliacion"] == "Duplicado"]
print(f"Registros marcados como 'Duplicado' por conciliación: {posibles_duplicados_negocio.shape[0]}")
posibles_duplicados_negocio

Registros marcados como 'Duplicado' por conciliación: 4


,transaccion_id,fecha_operacion,fecha_contable,empresa_id,empresa,cuenta_bancaria,banco,tipo_movimiento,categoria,proveedor_cliente,documento_ref,moneda,valor_bruto,impuesto_iva,valor_neto,metodo_pago,centro_costo,ciudad,pais,responsable,estado_conciliacion,fecha_vencimiento,dias_mora,nivel_riesgo,canal,observaciones,requiere_revision,motivo_revision,regla_signo_valida,regla_valor_neto_valida,regla_fechas_valida,regla_mora_valida
36,TRX-037,2026-04-25,2026-04-25,EMP-001,Nova Retail S.A.S.,1001-884422,Banco Capital,Egreso,Reembolso,Talento Humano,PAG-20260037,COP,-2746521,0.00,-2746521.00,Tarjeta credito,Comercial,Cali,Colombia,Diana Torres,Duplicado,2026-04-18,75,Medio,PSE,,True,Conciliacion no cerrada,True,True,True,True
73,TRX-074,2026-02-19,2026-02-19,EMP-001,Nova Retail S.A.S.,1001-884422,Banco Pacifico,Egreso,Impuestos,Sin registro,PAG-20260074,COP,-5408042,0.00,-5408042.00,Debito automatico,Operaciones,Cartagena,Colombia,Sin asignar,Duplicado,2026-02-17,135,Medio,Transferencia,,True,Conciliacion no cerrada,True,True,True,True
110,TRX-111,2026-06-10,2026-06-10,EMP-001,Nova Retail S.A.S.,1001-884422,Banco Nacional,Egreso,Credito bancario,Servicios Beta,PAG-20260111,COP,-8069563,0.00,-8069563.00,Transferencia local,Tecnologia,Medellin,Colombia,Carlos Pena,Duplicado,2026-06-13,19,Bajo,Tarjeta,,True,Conciliacion no cerrada; Monto alto,True,True,True,True
147,TRX-148,2026-04-06,2026-04-07,EMP-002,Logistica Prisma Ltda.,1002-553311,Banco Andino,Egreso,Arriendo,Arrendadora Centro,PAG-20260148,COP,-1331084,-252905.96,-1583989.96,Cheque,Finanzas,Barranquilla,Colombia,Felipe Mora,Duplicado,2026-04-14,79,Medio,Debito automatico,,True,Conciliacion no cerrada,True,True,True,True


**Decisión:** se dejan marcados

In [20]:
# No se eliminan, se dejan marcados para que quede visible en el análisis
df_limpio["bandera_duplicado_conciliacion"] = df_limpio["estado_conciliacion"] == "Duplicado"

# 7. Verificación final antes de exportar

In [21]:
print("Forma final:", df_limpio.shape)
print("\nNulos restantes:\n", df_limpio.isnull().sum().sum())
print("\nTipos de dato:\n", df_limpio.dtypes)
df_limpio.head()

Forma final: (150, 33)

Nulos restantes:
 0

Tipos de dato:
 transaccion_id                            object
fecha_operacion                   datetime64[ns]
fecha_contable                    datetime64[ns]
empresa_id                                object
empresa                                   object
cuenta_bancaria                           object
banco                                     object
tipo_movimiento                           object
categoria                                 object
proveedor_cliente                         object
documento_ref                             object
moneda                                    object
valor_bruto                                int64
impuesto_iva                             float64
valor_neto                               float64
metodo_pago                               object
centro_costo                              object
ciudad                                    object
pais                                      object
responsa

,transaccion_id,fecha_operacion,fecha_contable,empresa_id,empresa,cuenta_bancaria,banco,tipo_movimiento,categoria,proveedor_cliente,documento_ref,moneda,valor_bruto,impuesto_iva,valor_neto,metodo_pago,centro_costo,ciudad,pais,responsable,estado_conciliacion,fecha_vencimiento,dias_mora,nivel_riesgo,canal,observaciones,requiere_revision,motivo_revision,regla_signo_valida,regla_valor_neto_valida,regla_fechas_valida,regla_mora_valida,bandera_duplicado_conciliacion
0,TRX-001,2026-01-07,2026-01-07,EMP-002,Logistica Prisma Ltda.,1002-553311,Banco Capital,Egreso,Nomina,Servicios Beta,NOM-20260001,COP,-156933,0.00,-156933.00,Transferencia local,Comercial,Medellin,Colombia,Carlos Pena,Conciliado,2025-12-31,0,Bajo,PSE,,False,No aplica,True,True,True,True,False
1,TRX-002,2026-01-10,2026-01-10,EMP-003,Insumos Horizonte S.A.,1003-902244,Banco Pacifico,Egreso,Impuestos,Aduanas Global,PAG-20260002,COP,-228866,0.00,-228866.00,Tarjeta credito,Operaciones,Cali,Colombia,Diana Torres,Conciliado,2026-01-08,0,Bajo,Transferencia,,False,No aplica,True,True,True,True,False
2,TRX-003,2026-01-13,2026-01-13,EMP-001,Nova Retail S.A.S.,1001-884422,Banco Nacional,Egreso,Servicios publicos,Energia Urbana,PAG-20260003,COP,-300799,-57151.81,-357950.81,Cheque,Tecnologia,Barranquilla,Colombia,Felipe Mora,Conciliado,2026-01-16,0,Bajo,Tarjeta,,False,No aplica,True,True,True,True,False
3,TRX-004,2026-01-16,2026-01-17,EMP-002,Logistica Prisma Ltda.,1002-553311,Banco Andino,Egreso,Arriendo,Talento Humano,PAG-20260004,COP,-372732,-70819.08,-443551.08,Debito automatico,Finanzas,Cartagena,Colombia,Sin asignar,Pendiente,2026-01-24,159,Medio,Debito automatico,,True,Conciliacion no cerrada,True,True,True,True,False
4,TRX-005,2026-01-19,2026-01-19,EMP-003,Insumos Horizonte S.A.,1003-902244,Banco Capital,Ingreso,Credito bancario,Arrendadora Centro,FAC-20260005,COP,444665,0.00,444665.00,ACH,Sin asignar,Bogota,Colombia,Ana Ruiz,Conciliado,2026-02-01,0,Bajo,Caja,,False,No aplica,True,True,True,True,False


# 8. Exportar dataset limpio

Se guarda en `data/processed/`, nunca sobreescribiendo el archivo original.

In [23]:
RUTA_PROCESSED = "dataset_bancario_limpio.xlsx"
df_limpio.to_excel(RUTA_PROCESSED, index=False)
print(f"Dataset limpio exportado en: {RUTA_PROCESSED}")

Dataset limpio exportado en: dataset_bancario_limpio.xlsx


# 9. Resumen de decisiones tomadas

Se procesaron 150 transacciones del dataset bancario empresarial (dataset_bancario_empresarial.xlsx), estandarizando formatos de fecha y tipos de dato, se unificó texto en columnas categóricas, y se documentaron explícitamente los valores faltantes en 6 columnas (61 registros con al menos un campo sin capturar, marcados como "Sin asignar" o "Sin registro" según el caso).
Se validaron 5 reglas de negocio (consistencia de signo contable, coincidencia de valor neto, orden de fechas, y coherencia entre estado de conciliación y mora) — no se encontraron inconsistencias. El dataset limpio se exportó con el nombre dataset_limpio.xlsx sin alterar el archivo original.